# Imports

In [ ]:
from pathlib import Path
import pandas as pd

# load data

In [ ]:
csvs = [pd.read_csv(csv) for csv in Path("interactions_results").glob("*.csv")]

In [ ]:
df = pd.concat(csvs, ignore_index=True)

In [ ]:
pdb_paths = df.structure.unique()

In [ ]:
# use regex to parse i.e. "ASAP-0013249-001" from "MERS_x1029_ASAP-0013249-001_refine_min.pdb"
import re
def extract_asap_id(pdb_path):
    """
    Extracts the PDB ID from a file path.
    
    Args:
        pdb_path (str): The full path to the PDB file.
        
    Returns:
        str: The extracted PDB ID.
    """
    match = re.search(r'ASAP-\d{7}', str(pdb_path))
    if match:
        return match.group(0)
    return None

In [ ]:
example_path = "MERS_x1029_ASAP-0013249-001_refine_min.pdb"

In [ ]:
extract_asap_id(example_path)

In [ ]:
e2 = "MERS_x0139_ASAP-0008314-001_refine_min.pdb"

In [ ]:
# use regex to parse i.e. "x0139" from "MERS_x0139_ASAP-0008314-001_refine_min.pdb"
def extract_structure_name(pdb_path):
    """
    Extracts the structure name from a file path.
    
    Args:
        pdb_path (str): The full path to the PDB file.
        
    Returns:
        str: The extracted structure name.
    """
    match = re.search(r'_(x|P)\d+', str(pdb_path))
    if match:
        return match.group(0)[1:]  # Remove the leading underscore
    return None

In [ ]:
extract_structure_name(e2)

In [ ]:
def get_data_from_path_name(pdb_path):
    """
    Extracts the PDB ID and chain from a PDB file path.
    
    Args:
        pdb_path (str): The full path to the PDB file.
        
    Returns:
        tuple: A tuple containing the PDB ID and chain.
    """
    directory = pdb_path.parent
    parts = directory.name.split("_")
    variant = parts[0].upper()
    structure_type = "crystal" if "crystal" in parts else "docked"
    structure_name = extract_structure_name(pdb_path.name)
    asap_id = extract_asap_id(pdb_path.name)
    df = pd.DataFrame({
        "structure": [str(pdb_path)],
        "variant": [variant],
        "structure_type": [structure_type],
        "structure_name": [structure_name],
        "asap_id": [asap_id]
    })
    return df

In [ ]:
structure_data_dfs = [get_data_from_path_name(Path(pdb_path)) for pdb_path in pdb_paths]

In [ ]:
ddf = pd.concat(structure_data_dfs, ignore_index=True)

In [ ]:
df = df.merge(ddf, on="structure", how="inner")

In [ ]:
# save the merged dataframe
df.to_csv("merged_structure_data.csv", index=False)

## how many are missing?

# load dataframe

In [ ]:
df = pd.read_csv("merged_structure_data.csv")

# calculate fp at each level

In [ ]:
import sys
sys.path.append("/Users/alexpayne/Scientific_Projects/broad-spectrum-asap-paper/scripts")

In [ ]:
import plip_analysis_schema as pa

In [ ]:
df["interactions_csv"] = df.variant.apply(lambda x: x.lower()) + "_" + df.structure_type + "_" + df.report_id + "_interactions.csv"

In [ ]:
plint_reports = [pa.PLIntReport.from_csv(Path("/Users/alexpayne/Scientific_Projects/fragment-fold/science/20240820_add_plip_score/20250617_plip_analysis/interactions_results") / x) for x in df.interactions_csv.unique()]

In [ ]:
plint_reports[0].structure

In [ ]:
data = pd.concat([get_data_from_path_name(Path(pl.structure)) for pl in plint_reports])

In [ ]:
data["PlintReport"] = plint_reports

In [ ]:
# get all unique asap_ids
asap_ids = df.asap_id.unique()

In [ ]:
## split into 4 dataframes
mers_docked = data[(data.variant == "MERS")&(data.structure_type == "docked")]
sars2_docked = data[(data.variant == "SARS2")&(data.structure_type == "docked")]
mers_crystal = data[(data.variant == "MERS")&(data.structure_type == "crystal")]
sars2_crystal = data[(data.variant == "SARS2")&(data.structure_type == "crystal")]

In [ ]:
"ASAP-0013716"

In [ ]:
mers_docked[data.asap_id]

In [ ]:
def first_or_none(lst):
  if lst:  # Checks if the list is not empty
    return lst[0]
  else:
    return None

In [ ]:
def get_report_by_asap_id(data, variant, structure_type, asap_id):
    """
    Get the PLIntReport for a given variant, structure type, and ASAP ID.
    
    Args:
        data (pd.DataFrame): The dataframe containing the data.
        variant (str): The variant name (e.g., "MERS", "SARS2").
        structure_type (str): The structure type (e.g., "docked", "crystal").
        asap_id (str): The ASAP ID to filter by.
        
    Returns:
        PLIntReport: The corresponding PLIntReport object.
    """
    report = data[(data.variant == variant) & 
                  (data.structure_type == structure_type) & 
                  (data.asap_id == asap_id)]["PlintReport"].to_list()
    return first_or_none(report)

In [ ]:
from importlib import reload
reload(pa)
# Calculate scores
score_list = []
for asap_id in asap_ids:
    # filter data for this asap_id
    print(f"Processing ASAP ID: {asap_id}")
    mers_docked_report = get_report_by_asap_id(data, "MERS", "docked", asap_id)
    sars2_docked_report = get_report_by_asap_id(data, "SARS2", "docked", asap_id)
    mers_crystal_report = get_report_by_asap_id(data, "MERS", "crystal", asap_id)
    sars2_crystal_report = get_report_by_asap_id(data, "SARS2", "crystal", asap_id)
    
    crystal_reports = {
        "MERS": mers_crystal_report,
        "SARS2": sars2_crystal_report
    }
    docked_reports = {
        "MERS": mers_docked_report,
        "SARS2": sars2_docked_report
    }
    for level in pa.FingerprintLevel:
        for cname, creport in crystal_reports.items():
            if creport is None:
                print(f"No crystal report for {cname} with ASAP ID {asap_id}")
                continue
            for dname, dreport in docked_reports.items():
                if dreport is None:
                    print(f"No docked report for {dname} with ASAP ID {asap_id}")
                    continue
                name = f"{dname}_docked_to_{cname}_crystal_{level.value}"
                score = pa.InteractionScore.from_fingerprints(
                    creport, dreport, level
                )
                score_list.append({'Structure': name, **score.dict()})

In [ ]:
# Create dataframe and calculate ratios
df = pd.DataFrame.from_records(score_list)

In [ ]:
df["ratio_of_intersection"] = (
        df["number_of_interactions_in_intersection"] /
        df["number_of_interactions_in_reference"]
)
df["ratio_of_query"] = (
        df["number_of_interactions_in_query"] /
        df["number_of_interactions_in_reference"]
)

df["Docked_Variant"] = df.Structure.str.split('_').str[0]
df["Crystal_Variant"] = df.Structure.str.split('_').str[3]

# Save result
df.to_csv("results.csv")